In [13]:
import pandas as pd
import json
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, classification_report

In [ ]:
with open(r"typos.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

rows = []

for item in data:
    if not item.get("published", False):
        continue

    correct = item["key"]

    for incorrect in item.get("incorrectForms", []):
        rows.append({
            "incorrect": incorrect,
            "correct": correct
        })

df = pd.DataFrame(rows)

print("Total examples:", len(df))
print("Unique incorrect forms:", df["incorrect"].nunique())
print("Unique correct words:", df["correct"].nunique())

display(df.head(20))

Total examples: 185
Unique incorrect forms: 185
Unique correct words: 4


,incorrect,correct
0,مشمبا فرسزر,کیسه فریزر
1,مشمباها,کیسه فریزر
2,مشنبا فریر,کیسه فریزر
3,مشما فریرز,کیسه فریزر
4,موشمپا,کیسه فریزر
5,مشتبا فرسزر,کیسه فریزر
6,مشما فیلیزر,کیسه فریزر
7,مشمبا فیلیزر,کیسه فریزر
8,موشنبا,کیسه فریزر
9,مشما فریزر,کیسه فریزر


In [15]:
df = df.drop_duplicates(subset=["incorrect", "correct"])

# Remove incorrect forms with conflicting labels
ambiguous = df.groupby("incorrect")["correct"].nunique()
ambiguous = ambiguous[ambiguous > 1].index

df = df[~df["incorrect"].isin(ambiguous)].reset_index(drop=True)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["correct"]
)

print("Training examples:", len(train_df))
print("Testing examples:", len(test_df))

Training examples: 148
Testing examples: 37


In [16]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(
    train_df["incorrect"],
    train_df["correct"]
)

print("Training completed!")

Training completed!


In [17]:
y_true = test_df["correct"]
y_pred = model.predict(test_df["incorrect"])

print(f"Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%")

print(classification_report(
    y_true,
    y_pred,
    zero_division=0
))

Accuracy: 100.00%
              precision    recall  f1-score   support

   ما الشعیر       1.00      1.00      1.00        21
      نوشابه       1.00      1.00      1.00        11
        کشمش       1.00      1.00      1.00         1
  کیسه فریزر       1.00      1.00      1.00         4

    accuracy                           1.00        37
   macro avg       1.00      1.00      1.00        37
weighted avg       1.00      1.00      1.00        37



In [18]:
def correct_word(word):
    return model.predict([word])[0]

In [19]:
test_words = [
    "کشمیش",
    "کیشمش",
    "کیشمیش",
    "گشمش",
    "کیشمس"
]

for word in test_words:
    print(f"{word} -> {correct_word(word)}")

کشمیش -> کشمش
کیشمش -> کشمش
کیشمیش -> کشمش
گشمش -> کشمش
کیشمس -> کشمش
